<a href="https://colab.research.google.com/github/edwardsnj/pride-study-retrieval-cofest-2026/blob/main/notebooks/TextEmbedding%2BTFIDF_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#
# Import necessary modules
#

import sys, os, httpimport

with httpimport.github_repo("EdwardsLabProjects", "pride-study-retrieval-cofest-2026", ref="main"):
    from notebooks.lib import *

set_random_seed(state=None)

Version: 1.0.29
Using random seed: 8495459


In [2]:
#
# Get embeddings and known true positives
#

md,emb = embeddings("openai-3-small")
embdim = emb.shape[0]
tp,tn = knownstudies()

allacc = set(emb.columns)
tp &= allacc
tn &= allacc

assert len(tp & tn) == 0, "TP and TN should not intersect!"

# Print out some details...
print(f"Embeddings: {emb.shape}")
print(f"True Pos: {len(tp)}")
print(f"True Neg: {len(tn)}")

# allacc,avgemb = select_by_embedding_proximity(tp,emb,1000)
# print(f"All Acc: {len(allacc)}")

Embeddings: (1536, 36696)
True Pos: 86
True Neg: 47


In [3]:
#
# Create train/test split
#
TESTFRAC = 0.2
BACKGROUND_OVERSAMPLE = 25

train_acc, train_y, test_acc, test_y = split_train_test(allacc, tp, tn,
    test_size=TESTFRAC, bgsize=BACKGROUND_OVERSAMPLE)


In [4]:
# Build the TF-IDF features from train data only,
# but build feature vectors for all documents

from sklearn.feature_extraction import text
stop_words = list(set(list(text.ENGLISH_STOP_WORDS) + """

""".split()))

tfidf_extra_args = dict(positive_only=False, # Use only positive set for word selection
                        min_df=1, # minimum documents word is in
                        max_df=1.0, # maximum documents word is in
                        max_features=None, # max number of words to use
                        stop_words=stop_words) # words to ignore

tfidf,tfidf_model = create_tfidf_features(md, train_acc, train_y,
                                          test_acc, **tfidf_extra_args)

print(f"TF-IDF features shape: {tfidf.shape}")

TF-IDF features shape: (35760, 2236)


In [5]:
logreg_extra_args = dict(class_weight='balanced',C=10.0,
                         penalty='l1', solver='liblinear',
                         use_embed=True, use_tfidf=True)
trained_model = train_document_classifier(emb, tfidf,
                                          train_acc, train_y,
                                          test_acc, test_y,
                                          **logreg_extra_args)
print("NZ Coeffs:",sum(*[1*(xi!=0) for xi in trained_model.coef_]))

Training data shape: (1788, 37296) (Positives: 68, Negatives: 1720)
Testing data shape: (448, 37296) (Positives: 18, Negatives: 430)

--- Model Evaluation ---
Accuracy: 0.9777

Classification Report:
                precision    recall  f1-score   support

Background (0)       1.00      0.98      0.99       430
 Seed-like (1)       0.65      0.94      0.77        18

      accuracy                           0.98       448
     macro avg       0.83      0.96      0.88       448
  weighted avg       0.98      0.98      0.98       448

NZ Coeffs: 33


In [6]:
nzemb,nztfidf,topfeat = top_features(trained_model,tfidf_model,nembed=embdim,**logreg_extra_args)
print(f"Number of sem. embedding dimensions with non-zero coefficients: {nzemb}")
print(f"Number of TF-IDF dimensions with non-zero coefficients: {nztfidf}")
print("\nTop 20 TF-IDF Features:")
display(topfeat.head(20))

# show_top_feature_examples(topfeat, md)


Number of sem. embedding dimensions with non-zero coefficients: 0
Number of TF-IDF dimensions with non-zero coefficients: 33

Top 20 TF-IDF Features:


,Feature,Coefficient
25855,pngase,131.013491
6124,asn,34.442059
8255,cat,-28.102655
15500,glycosite,21.690744
15511,glycosylation,21.388043
7774,bx,17.345581
18923,kegg,15.768837
10869,deamidation,13.459466
23064,nmpo,12.963980
15509,glycosylated,12.613483


In [7]:
results = score_all_studies(trained_model, emb, md, tfidf_model, train_acc, tp, tn)
display(results)

Scoring studies: 100%|##########| 74/74 [00:42<00:00,  1.75it/s]


,prideacc,title,probability,in_training,true_positive,true_negative
0,PXD037140,H11-induced immune protection is predominantly...,1.000000,False,False,False
1,PXD023379,N-Glycome and N-Glycoproteome of a Hematophago...,1.000000,True,True,False
2,PXD010283,Sepioloidea lineolata MSMS,1.000000,False,False,False
3,PXD009808,Raw N-glycan MALDI mass spectrometry imaging d...,1.000000,False,False,False
4,PXD004034,Glycoproteomes of naive and LPS stimulated hum...,1.000000,False,True,False
5,PXD038280,Lectin-based affinity enrichment and character...,1.000000,False,True,False
6,PXD006400,OVA N-glycans PGC-LC-MSMS - Large-scale ident...,1.000000,False,False,False
7,PXD012655,Integrated proteomic and N-glycoproteomic anal...,1.000000,False,True,False
8,PXD015389,Sialic acid-mediated mechanical activation of ...,1.000000,False,False,False
9,PXD024767,1. Site-specific O-glycosylation analysis of S...,1.000000,False,False,False
